# Gemma 3 12B — Extract Text Decoder + Export Vision ONNX for TRT-LLM

**Purpose**: Prepare the merged VLM model for local TRT-LLM INT4 engine build.

**Input**: Merged VLM on Google Drive at `/content/drive/MyDrive/gemma3-12b-legal-merged-16bit`

**Outputs**:
- Text-only `Gemma3ForCausalLM` (~21 GB) — for TRT-LLM `convert_checkpoint.py` + `trtllm-build`
- SigLIP vision encoder ONNX (~1.5 GB) — for `trtexec` engine build
- Projector ONNX (~50 MB) — for `trtexec` engine build

**Why not quantize here?** TRT-LLM `convert_checkpoint.py` and `trtllm-build` must run in the
same TRT-LLM version. Our Docker container (`Dockerfile.trtllm`, v0.21.0) is the controlled
environment. Doing conversion here risks version mismatch.

**Hardware**: A100 (40GB) recommended. T4 works but slower.

**Time**: ~15-20 minutes total

---

## Architecture
```
Merged VLM (Gemma3ForConditionalGeneration)
  ├── language_model (Gemma3ForCausalLM)     → Extract → text-only HF model
  ├── vision_tower (SigLIP-SO400M)           → Export  → siglip_vision.onnx
  └── multi_modal_projector (Linear layers)   → Export  → gemma_projector.onnx
```

## 1. Install Dependencies

In [ ]:
!pip install -q transformers>=4.45.0 safetensors torch onnx accelerate

import torch
import sys
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM: {vram_gb:.1f} GB")

import psutil
ram_gb = psutil.virtual_memory().total / 1024**3
print(f"System RAM: {ram_gb:.1f} GB")

## 2. Mount Drive + Find Merged Model

In [ ]:
import os
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Check known paths for the merged model
MERGED_CANDIDATES = [
    '/content/drive/MyDrive/gemma3-12b-legal-merged-16bit',
    '/content/drive/MyDrive/COLAB_PACKAGE/gemma3-12b-legal-merged-16bit',
    '/content/gemma3-12b-legal-merged-16bit',
]

MERGED_DIR = None
for candidate in MERGED_CANDIDATES:
    p = Path(candidate)
    if p.exists() and (p / 'config.json').exists():
        MERGED_DIR = str(p)
        break

if not MERGED_DIR:
    raise FileNotFoundError(
        "Merged model not found! Expected at:\n"
        + "\n".join(f"  {c}" for c in MERGED_CANDIDATES)
        + "\nRun Gemma3_12B_Merge_and_TRT_Export.ipynb first."
    )

print(f"Merged model: {MERGED_DIR}")

# List contents
total_gb = 0
for f in sorted(Path(MERGED_DIR).iterdir()):
    if f.is_file():
        size_mb = f.stat().st_size / (1024**2)
        total_gb += size_mb / 1024
        print(f"  {f.name:45s} {size_mb:>8.1f} MB")
print(f"{'':45s} {'─'*12}")
print(f"  {'Total':45s} {total_gb:>7.1f} GB")

## 3. Analyze VLM Config

Verify this is a VLM (`Gemma3ForConditionalGeneration`) and inspect the text/vision split.

In [ ]:
import json

with open(f'{MERGED_DIR}/config.json') as f:
    vlm_config = json.load(f)

arch = vlm_config.get('architectures', ['unknown'])[0]
print(f"Architecture: {arch}")

if 'Conditional' not in arch:
    print("\nThis is already a text-only model (Gemma3ForCausalLM).")
    print("No extraction needed — skip to Cell 7 (Save to Drive).")
    IS_VLM = False
else:
    IS_VLM = True
    print("\nVLM detected — will extract text decoder + export vision components.")

    # Text config
    text_cfg = vlm_config.get('text_config', {})
    print(f"\nText Decoder:")
    print(f"  hidden_size:    {text_cfg.get('hidden_size', vlm_config.get('hidden_size', '?'))}")
    print(f"  num_layers:     {text_cfg.get('num_hidden_layers', vlm_config.get('num_hidden_layers', '?'))}")
    print(f"  num_heads:      {text_cfg.get('num_attention_heads', vlm_config.get('num_attention_heads', '?'))}")
    print(f"  vocab_size:     {text_cfg.get('vocab_size', vlm_config.get('vocab_size', '?'))}")

    # Vision config
    vis_cfg = vlm_config.get('vision_config', {})
    if vis_cfg:
        print(f"\nVision Encoder:")
        print(f"  model_type:     {vis_cfg.get('model_type', '?')}")
        print(f"  hidden_size:    {vis_cfg.get('hidden_size', '?')}")
        print(f"  image_size:     {vis_cfg.get('image_size', '?')}")
        print(f"  patch_size:     {vis_cfg.get('patch_size', '?')}")
        num_patches = (vis_cfg.get('image_size', 384) // vis_cfg.get('patch_size', 14)) ** 2
        print(f"  num_patches:    {num_patches}")
    else:
        print("\nNo vision_config found in config.json")

print(f"\nFull config keys: {list(vlm_config.keys())}")

## 4. Load VLM + Extract Text-Only Model

Load the full VLM, extract `model.language_model` as standalone `Gemma3ForCausalLM`,
and save to Colab local storage.

**Memory**: Loads ~24GB BF16 on GPU (A100 40GB) or CPU.

In [ ]:
import torch
import gc
from transformers import AutoProcessor, AutoTokenizer

TEXT_ONLY_DIR = '/content/gemma3-12b-legal-text-only'
ONNX_DIR = '/content/trt_onnx_exports'
os.makedirs(TEXT_ONLY_DIR, exist_ok=True)
os.makedirs(ONNX_DIR, exist_ok=True)

if IS_VLM:
    from transformers import Gemma3ForConditionalGeneration

    print("Loading VLM (this takes 2-5 minutes)...")
    vlm = Gemma3ForConditionalGeneration.from_pretrained(
        MERGED_DIR,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    print(f"VLM loaded on: {vlm.device if hasattr(vlm, 'device') else 'auto'}")

    # ── Extract text decoder ──
    print("\nExtracting text decoder (language_model)...")
    text_model = vlm.language_model

    # Move to CPU for saving (frees GPU for ONNX export later)
    text_model = text_model.to('cpu')

    # Ensure config has correct architecture tag
    if not hasattr(text_model.config, 'architectures') or not text_model.config.architectures:
        text_model.config.architectures = ['Gemma3ForCausalLM']
    elif 'Gemma3ForCausalLM' not in text_model.config.architectures:
        text_model.config.architectures = ['Gemma3ForCausalLM']

    print(f"Text model config arch: {text_model.config.architectures}")
    print(f"Text model params: {sum(p.numel() for p in text_model.parameters()) / 1e9:.1f}B")

    print(f"\nSaving text-only model to {TEXT_ONLY_DIR}/ ...")
    text_model.save_pretrained(
        TEXT_ONLY_DIR,
        safe_serialization=True,
        max_shard_size="5GB",
    )

    # Copy tokenizer from original
    try:
        tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
        tokenizer.save_pretrained(TEXT_ONLY_DIR)
        print("Tokenizer saved.")
    except Exception as e:
        print(f"Tokenizer copy warning: {e}")
        print("Tokenizer can be loaded from base model later.")

    # Verify
    text_size_gb = sum(
        f.stat().st_size for f in Path(TEXT_ONLY_DIR).rglob('*') if f.is_file()
    ) / (1024**3)
    safetensors_count = len(list(Path(TEXT_ONLY_DIR).glob('*.safetensors')))
    print(f"\nText-only model saved: {text_size_gb:.1f} GB ({safetensors_count} shards)")

    # Keep VLM in memory for ONNX exports (cells 5-6)
    # Move vision components to GPU for export
    del text_model
    gc.collect()
    torch.cuda.empty_cache()
    print("Text decoder extracted. VLM still loaded for vision export.")

else:
    print("Model is already text-only. Copying to local...")
    import shutil
    if os.path.exists(TEXT_ONLY_DIR):
        shutil.rmtree(TEXT_ONLY_DIR)
    shutil.copytree(MERGED_DIR, TEXT_ONLY_DIR)
    text_size_gb = sum(
        f.stat().st_size for f in Path(TEXT_ONLY_DIR).rglob('*') if f.is_file()
    ) / (1024**3)
    print(f"Copied: {text_size_gb:.1f} GB")

## 5. Export SigLIP Vision Encoder to ONNX

SigLIP-SO400M is a standard ViT that exports cleanly to ONNX.
On local machine, `trtexec` converts ONNX → TRT engine (FP16, sm_86).

**Skip this cell if the model is text-only.**

In [ ]:
import torch
import onnx

if not IS_VLM:
    print("Text-only model — no vision encoder to export. Skipping.")
else:
    SIGLIP_ONNX = f"{ONNX_DIR}/siglip_vision.onnx"

    print("Extracting SigLIP vision encoder...")
    vision_tower = vlm.vision_tower
    vision_tower = vision_tower.to('cuda').eval()

    # Get image size from config
    vis_cfg = vlm_config.get('vision_config', {})
    img_size = vis_cfg.get('image_size', 384)
    vision_hidden = vis_cfg.get('hidden_size', 1152)
    patch_size = vis_cfg.get('patch_size', 14)
    num_patches = (img_size // patch_size) ** 2

    print(f"  Image size: {img_size}x{img_size}")
    print(f"  Patch size: {patch_size}")
    print(f"  Num patches: {num_patches}")
    print(f"  Hidden dim: {vision_hidden}")
    print(f"  Params: {sum(p.numel() for p in vision_tower.parameters()) / 1e6:.0f}M")

    # Create dummy input
    dummy_pixels = torch.randn(1, 3, img_size, img_size, dtype=torch.float32, device='cuda')

    # Test forward pass first
    print("\nTesting forward pass...")
    with torch.no_grad():
        test_out = vision_tower(dummy_pixels)
        # Handle different output types
        if hasattr(test_out, 'last_hidden_state'):
            test_tensor = test_out.last_hidden_state
        elif isinstance(test_out, tuple):
            test_tensor = test_out[0]
        else:
            test_tensor = test_out
        print(f"  Output shape: {test_tensor.shape}")
        print(f"  Output dtype: {test_tensor.dtype}")

    # Export to ONNX
    print(f"\nExporting to ONNX: {SIGLIP_ONNX}")

    # Wrap vision tower to return just the tensor
    class VisionWrapper(torch.nn.Module):
        def __init__(self, tower):
            super().__init__()
            self.tower = tower

        def forward(self, pixel_values):
            out = self.tower(pixel_values)
            if hasattr(out, 'last_hidden_state'):
                return out.last_hidden_state
            elif isinstance(out, tuple):
                return out[0]
            return out

    wrapper = VisionWrapper(vision_tower).eval()

    torch.onnx.export(
        wrapper,
        dummy_pixels,
        SIGLIP_ONNX,
        input_names=['pixel_values'],
        output_names=['image_features'],
        dynamic_axes={
            'pixel_values': {0: 'batch'},
            'image_features': {0: 'batch'},
        },
        opset_version=17,
        do_constant_folding=True,
    )

    # Verify ONNX
    onnx_model = onnx.load(SIGLIP_ONNX)
    onnx.checker.check_model(onnx_model, full_check=True)
    siglip_size_mb = os.path.getsize(SIGLIP_ONNX) / (1024**2)
    print(f"SigLIP ONNX exported: {siglip_size_mb:.0f} MB")
    print(f"ONNX check: PASSED")

    del wrapper, vision_tower, test_out, test_tensor, dummy_pixels
    torch.cuda.empty_cache()

## 6. Export Projector to ONNX

The projection layers bridge SigLIP output (1152-dim) to Gemma3 text input (3840-dim).
Tiny model (~50MB).

**Skip this cell if the model is text-only.**

In [ ]:
import torch
import onnx

if not IS_VLM:
    print("Text-only model — no projector to export. Skipping.")
else:
    PROJECTOR_ONNX = f"{ONNX_DIR}/gemma_projector.onnx"

    print("Extracting multi-modal projector...")
    projector = vlm.multi_modal_projector
    projector = projector.to('cuda').eval()

    vis_cfg = vlm_config.get('vision_config', {})
    vision_hidden = vis_cfg.get('hidden_size', 1152)
    img_size = vis_cfg.get('image_size', 384)
    patch_size = vis_cfg.get('patch_size', 14)
    num_patches = (img_size // patch_size) ** 2

    print(f"  Input: ({num_patches}, {vision_hidden}) per image")
    print(f"  Params: {sum(p.numel() for p in projector.parameters()) / 1e6:.1f}M")

    # Dummy input: batch of vision features
    dummy_features = torch.randn(
        1, num_patches, vision_hidden,
        dtype=torch.float32, device='cuda'
    )

    # Test forward pass
    print("\nTesting forward pass...")
    with torch.no_grad():
        proj_out = projector(dummy_features)
        if isinstance(proj_out, tuple):
            proj_out = proj_out[0]
        print(f"  Output shape: {proj_out.shape}")
        print(f"  Output dtype: {proj_out.dtype}")

    # Export
    print(f"\nExporting to ONNX: {PROJECTOR_ONNX}")

    torch.onnx.export(
        projector,
        dummy_features,
        PROJECTOR_ONNX,
        input_names=['image_features'],
        output_names=['projected_tokens'],
        dynamic_axes={
            'image_features': {0: 'batch'},
            'projected_tokens': {0: 'batch'},
        },
        opset_version=17,
        do_constant_folding=True,
    )

    # Verify
    onnx_model = onnx.load(PROJECTOR_ONNX)
    onnx.checker.check_model(onnx_model, full_check=True)
    proj_size_mb = os.path.getsize(PROJECTOR_ONNX) / (1024**2)
    print(f"Projector ONNX exported: {proj_size_mb:.1f} MB")
    print(f"ONNX check: PASSED")

    # Free VLM memory — no longer needed
    del projector, proj_out, dummy_features, vlm
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    print("\nVLM unloaded. GPU memory freed.")

## 7. Verify Text-Only Model Loads Standalone

Quick sanity check: load the extracted text-only model as `Gemma3ForCausalLM`
to confirm TRT-LLM's `convert_checkpoint.py` will accept it.

In [ ]:
import torch
import json
from pathlib import Path

# Check saved config
with open(f'{TEXT_ONLY_DIR}/config.json') as f:
    text_config = json.load(f)

arch = text_config.get('architectures', ['unknown'])[0]
print(f"Saved architecture: {arch}")
print(f"model_type: {text_config.get('model_type', '?')}")
print(f"hidden_size: {text_config.get('hidden_size', '?')}")
print(f"num_hidden_layers: {text_config.get('num_hidden_layers', '?')}")
print(f"vocab_size: {text_config.get('vocab_size', '?')}")

# Verify no vision keys leaked into the text model
print("\nChecking for leaked vision keys...")
from safetensors import safe_open
leaked = []
for sf in sorted(Path(TEXT_ONLY_DIR).glob('*.safetensors')):
    with safe_open(str(sf), framework='pt') as f:
        for key in f.keys():
            if 'vision' in key.lower() or 'projector' in key.lower():
                leaked.append(key)

if leaked:
    print(f"  WARNING: {len(leaked)} vision keys found in text model:")
    for k in leaked[:5]:
        print(f"    {k}")
    print("  These will cause convert_checkpoint.py to fail.")
    print("  Manual tensor filtering needed.")
else:
    print("  No vision keys found. Clean text-only model.")

# Quick load test (just config, not full weights — saves time)
from transformers import AutoConfig
try:
    test_config = AutoConfig.from_pretrained(TEXT_ONLY_DIR)
    print(f"\nAutoConfig loads: {type(test_config).__name__}")
    print(f"Config valid: YES")
except Exception as e:
    print(f"\nAutoConfig failed: {e}")
    print("May need manual config fixup.")

# File listing
print(f"\nText-only model contents:")
total_gb = 0
for f in sorted(Path(TEXT_ONLY_DIR).iterdir()):
    if f.is_file():
        size_mb = f.stat().st_size / (1024**2)
        total_gb += size_mb / 1024
        print(f"  {f.name:45s} {size_mb:>8.1f} MB")
print(f"  {'Total':45s} {total_gb:>7.1f} GB")

## 8. Save All Artifacts to Google Drive

Saves:
- Text-only model (~21 GB) → for `convert_checkpoint.py` + `trtllm-build`
- SigLIP ONNX (~1.5 GB) → for `trtexec`
- Projector ONNX (~50 MB) → for `trtexec`
- Manifest JSON with metadata

In [ ]:
import os
import shutil
import json
import hashlib
from pathlib import Path
from datetime import datetime

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_ARTIFACTS = '/content/drive/MyDrive/gemma3-12b-legal-trt-artifacts'

print(f"Saving artifacts to: {DRIVE_ARTIFACTS}")
print(f"This takes 10-20 minutes for ~22 GB...\n")

# Create directory structure
os.makedirs(f'{DRIVE_ARTIFACTS}/text_only_model', exist_ok=True)
os.makedirs(f'{DRIVE_ARTIFACTS}/onnx', exist_ok=True)

# ── Copy text-only model ──
print("Copying text-only model...")
text_dest = f'{DRIVE_ARTIFACTS}/text_only_model'
if os.path.exists(text_dest):
    shutil.rmtree(text_dest)
shutil.copytree(TEXT_ONLY_DIR, text_dest, dirs_exist_ok=True)
text_gb = sum(f.stat().st_size for f in Path(text_dest).rglob('*') if f.is_file()) / (1024**3)
print(f"  Text model: {text_gb:.1f} GB")

# ── Copy ONNX files ──
onnx_files = {}
siglip_src = f'{ONNX_DIR}/siglip_vision.onnx'
proj_src = f'{ONNX_DIR}/gemma_projector.onnx'

if os.path.exists(siglip_src):
    print("Copying SigLIP ONNX...")
    shutil.copy2(siglip_src, f'{DRIVE_ARTIFACTS}/onnx/siglip_vision.onnx')
    siglip_mb = os.path.getsize(siglip_src) / (1024**2)
    print(f"  SigLIP: {siglip_mb:.0f} MB")
    onnx_files['siglip_vision.onnx'] = siglip_mb

if os.path.exists(proj_src):
    print("Copying Projector ONNX...")
    shutil.copy2(proj_src, f'{DRIVE_ARTIFACTS}/onnx/gemma_projector.onnx')
    proj_mb = os.path.getsize(proj_src) / (1024**2)
    print(f"  Projector: {proj_mb:.1f} MB")
    onnx_files['gemma_projector.onnx'] = proj_mb

# ── Create manifest ──
manifest = {
    'created': datetime.now().isoformat(),
    'source_model': MERGED_DIR,
    'source_arch': vlm_config.get('architectures', ['unknown'])[0],
    'is_vlm': IS_VLM,
    'text_model': {
        'path': 'text_only_model/',
        'arch': 'Gemma3ForCausalLM',
        'size_gb': round(text_gb, 1),
        'dtype': 'bfloat16',
        'hidden_size': text_config.get('hidden_size', 3840),
        'num_layers': text_config.get('num_hidden_layers', 48),
        'vocab_size': text_config.get('vocab_size', 262144),
    },
    'onnx_models': onnx_files,
    'next_steps': [
        'Download text_only_model/ to local machine',
        'Download onnx/ files to local machine',
        'In Docker (TRT-LLM v0.21.0): python3 examples/gemma/convert_checkpoint.py --ckpt-type hf --model-dir /models/text_only --use-weight-only-with-precision int4 --dtype bfloat16 --world-size 1 --output-model-dir /models/int4_checkpoint',
        'In Docker: trtllm-build --checkpoint_dir /models/int4_checkpoint --gemm_plugin auto --gpt_attention_plugin auto --max_batch_size 4 --max_input_len 2048 --max_seq_len 4096 --output_dir /models/engine_int4',
        'In Docker: trtexec --onnx=/models/siglip_vision.onnx --saveEngine=/models/siglip_vision.engine --fp16',
        'In Docker: trtexec --onnx=/models/gemma_projector.onnx --saveEngine=/models/gemma_projector.engine --fp16',
    ],
}

with open(f'{DRIVE_ARTIFACTS}/export_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

# ── Summary ──
total_gb = sum(
    f.stat().st_size for f in Path(DRIVE_ARTIFACTS).rglob('*') if f.is_file()
) / (1024**3)
print(f"\n{'='*60}")
print(f"All artifacts saved to Google Drive: {total_gb:.1f} GB")
print(f"Path: {DRIVE_ARTIFACTS}")
print(f"{'='*60}")
print(f"\nDownload from: https://drive.google.com/")
print(f"Folder: My Drive / gemma3-12b-legal-trt-artifacts/")

## 9. Next Steps (On Local Machine)

Download the artifacts from Google Drive (~22 GB total), then build engines locally.

### Download
```
From: My Drive / gemma3-12b-legal-trt-artifacts/
To:   c:\Users\james\Videos\deeds-web-app\trt_artifacts\
```

### Build Engines in Docker (RTX 3060 Ti)
```bash
# 1. Build Docker image
docker build -f Dockerfile.trtllm -t legal-ai-trtllm .

# 2. Start container with GPU + model mounts
docker run --gpus all -it \
  -v ./trt_artifacts/text_only_model:/models/text_only:ro \
  -v ./trt_artifacts/onnx:/models/onnx:ro \
  -v ./engines:/models/engines \
  legal-ai-trtllm bash

# 3. Inside container — convert text model to INT4 checkpoint
cd /workspace/tensorrt-llm
python3 examples/gemma/convert_checkpoint.py \
  --ckpt-type hf \
  --model-dir /models/text_only \
  --use-weight-only-with-precision int4 \
  --dtype bfloat16 \
  --world-size 1 \
  --output-model-dir /models/int4_checkpoint

# 4. Build TRT engine (compiles for sm_86 / RTX 3060 Ti)
trtllm-build \
  --checkpoint_dir /models/int4_checkpoint \
  --gemm_plugin auto \
  --gpt_attention_plugin auto \
  --max_batch_size 4 \
  --max_input_len 2048 \
  --max_seq_len 4096 \
  --output_dir /models/engines/gemma3_12b_int4

# 5. Build SigLIP engine
trtexec \
  --onnx=/models/onnx/siglip_vision.onnx \
  --saveEngine=/models/engines/siglip_vision.engine \
  --fp16 \
  --optShapes=pixel_values:1x3x384x384 \
  --maxShapes=pixel_values:4x3x384x384

# 6. Build Projector engine
trtexec \
  --onnx=/models/onnx/gemma_projector.onnx \
  --saveEngine=/models/engines/gemma_projector.engine \
  --fp16
```

### Deploy via Triton
```bash
docker compose -f docker-compose.triton.yml up triton-legal-ai -d
curl http://localhost:8099/v2/health/ready
```

See `next_steps/TRT_ENGINE_BUILD_STEPS.md` for full details.